In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.models as models

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

In [10]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5),
            nn.ReLU(),
            nn.AvgPool2d(2),
            nn.Conv2d(6, 16, 5),
            nn.ReLU(),
            nn.AvgPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*53*53, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [11]:
def get_model(name):
    if name == "AlexNet":
        model = models.alexnet(pretrained=True)
        model.classifier[6] = nn.Linear(4096, 10)

    elif name == "VGG":
        model = models.vgg16(pretrained=True)
        model.classifier[6] = nn.Linear(4096, 10)

    elif name == "ResNet50":
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(2048, 10)

    elif name == "ResNet101":
        model = models.resnet101(pretrained=True)
        model.fc = nn.Linear(2048, 10)

    elif name == "MobileNet":
        model = models.mobilenet_v2(pretrained=True)
        model.classifier[1] = nn.Linear(1280, 10)

    elif name == "EfficientNet":
        model = models.efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(1280, 10)

    elif name == "Inception":
        model = models.inception_v3(pretrained=True)
        model.fc = nn.Linear(2048, 10)

    return model.to(device)

In [12]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        _, pred = out.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()

    return 100 * correct / total


def test_epoch(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return 100 * correct / total

In [13]:
model = LeNet5().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    train_epoch(model, trainloader, optimizer, criterion)

print("LeNet Test Accuracy:", test_epoch(model, testloader))

LeNet Test Accuracy: 57.17


In [14]:
model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    train_epoch(model, trainloader, optimizer, criterion)

print("AlexNet Test Accuracy:", test_epoch(model, testloader))

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 205MB/s]


AlexNet Test Accuracy: 73.83


In [16]:
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# FREEZE convolution layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
model.classifier[6] = nn.Linear(4096, 10)
model = model.to(device)

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):   # 3 is enough for comparison
    train_epoch(model, trainloader, optimizer, criterion)

print("VGG Test Accuracy:", test_epoch(model, testloader))

VGG Test Accuracy: 83.56


In [17]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(2048, 10)
model = model.to(device)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    train_epoch(model, trainloader, optimizer, criterion)

print("ResNet50 Test Accuracy:", test_epoch(model, testloader))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 139MB/s]


ResNet50 Test Accuracy: 81.49


In [18]:
model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(2048, 10)
model = model.to(device)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    train_epoch(model, trainloader, optimizer, criterion)

print("ResNet101 Test Accuracy:", test_epoch(model, testloader))

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 169MB/s]


ResNet101 Test Accuracy: 85.47


In [19]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(1280, 10)
model = model.to(device)

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    train_epoch(model, trainloader, optimizer, criterion)

print("EfficientNet Test Accuracy:", test_epoch(model, testloader))

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 201MB/s]


EfficientNet Test Accuracy: 80.57


In [20]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(1280, 10)
model = model.to(device)

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    train_epoch(model, trainloader, optimizer, criterion)

print("MobileNet Test Accuracy:", test_epoch(model, testloader))

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 205MB/s]


MobileNet Test Accuracy: 77.06


In [23]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Inception requires 299x299
inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Re-load dataset ONLY for Inception
trainset_inception = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=inception_transform
)

testset_inception = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=inception_transform
)

trainloader_inception = DataLoader(
    trainset_inception, batch_size=16, shuffle=True
)

testloader_inception = DataLoader(
    testset_inception, batch_size=16, shuffle=False
)

In [25]:
import torch.nn as nn
import torchvision.models as models

model = models.inception_v3(
    weights=models.Inception_V3_Weights.DEFAULT
)

model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [26]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [28]:
def train_epoch_inception(model, loader, optimizer, criterion):
    model.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        outputs = model(x)

        # FIX: Inception returns InceptionOutputs
        if hasattr(outputs, "logits"):
            outputs = outputs.logits

        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

for epoch in range(2):
    train_epoch_inception(model, trainloader_inception, optimizer, criterion)

model.eval()
acc = test_epoch(model, testloader_inception)
print("Inception Test Accuracy:", acc)

Inception Test Accuracy: 85.08


In [29]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss

In [30]:
class ArcFaceLoss(nn.Module):
    def __init__(self, margin=0.5, scale=30):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, labels):
        logits = logits / torch.norm(logits, dim=1, keepdim=True)
        logits = logits * self.scale
        return self.ce(logits, labels)

In [31]:
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

def train_bce(model, loader):
    model.train()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        y_onehot = torch.zeros(y.size(0), 10).to(device)
        y_onehot.scatter_(1, y.unsqueeze(1), 1)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y_onehot)
        loss.backward()
        optimizer.step()

        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total

In [32]:
for epoch in range(10):
    train_acc = train_bce(model, trainloader)

test_acc = test_epoch(model, testloader)
print("VGG | BCE | Train:", train_acc, "Test:", test_acc)

KeyboardInterrupt: 

In [33]:
model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)
model = model.to(device)

optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = FocalLoss(gamma=2)

In [34]:
for epoch in range(20):
    train_epoch(model, trainloader, optimizer, criterion)

test_acc = test_epoch(model, testloader)
print("AlexNet | Focal | Test:", test_acc)

AlexNet | Focal | Test: 90.55


In [35]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(2048, 10)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = ArcFaceLoss()

In [ ]:
for epoch in range(15):
    train_epoch(model, trainloader, optimizer, criterion)

test_acc = test_epoch(model, testloader)
print("ResNet | ArcFace | Test:", test_acc)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def extract_features_fast(model, loader, max_samples=1000):
    model.eval()
    feats, labs = [], []
    count = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            # Generic CNN feature extraction
            if hasattr(model, "features"):   # VGG / AlexNet
                f = model.features(x)
                f = f.view(f.size(0), -1)
            else:                             # ResNet
                f = model.avgpool(model.layer4(model.layer3(
                    model.layer2(model.layer1(
                        model.relu(model.bn1(model.conv1(x)))
                    ))
                )))
                f = f.view(f.size(0), -1)

            feats.append(f.cpu())
            labs.append(y)
            count += x.size(0)

            if count >= max_samples:
                break

    return torch.cat(feats).numpy(), torch.cat(labs).numpy()

In [ ]:
features_bce, labels_bce = extract_features_fast(
    model, testloader, max_samples=800
)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
bce_2d = tsne.fit_transform(features_bce)

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(bce_2d[:,0], bce_2d[:,1],
            c=labels_bce, cmap="tab10", s=10)
plt.title("t-SNE Feature Clustering (BCE Loss)")
plt.colorbar()
plt.show()

In [ ]:
features_arc, labels_arc = extract_features_fast(
    model, testloader, max_samples=800
)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
arc_2d = tsne.fit_transform(features_arc)

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(arc_2d[:,0], arc_2d[:,1],
            c=labels_arc, cmap="tab10", s=10)
plt.title("t-SNE Feature Clustering (ArcFace Loss)")
plt.colorbar()
plt.show()